In [1]:
pwd

'C:\\Users\\devan\\OneDrive\\Desktop\\Devansh_S\\Machine_Learning\\CINEIQ\\notebooks'

In [2]:
import pandas as pd

movies = pd.read_csv("../data/raw/movies.csv")

movies["genres_clean"] = movies["genres"].str.replace("|", " ", regex=False)
movies["year"] = movies["title"].str.extract(r"\((\d{4})\)")
movies["year"] = pd.to_numeric(movies["year"], errors="coerce")

movies.head()

,movieId,title,genres,genres_clean,year
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,Adventure Animation Children Comedy Fantasy,1995.0
1,2,Jumanji (1995),Adventure|Children|Fantasy,Adventure Children Fantasy,1995.0
2,3,Grumpier Old Men (1995),Comedy|Romance,Comedy Romance,1995.0
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,Comedy Drama Romance,1995.0
4,5,Father of the Bride Part II (1995),Comedy,Comedy,1995.0


In [4]:
# Load movies data
movies = pd.read_csv("../data/raw/movies.csv")

# Clean genres
movies["genres_clean"] = movies["genres"].str.replace("|", " ", regex=False)

# Use smaller data to avoid MemoryError
movies_small = movies.head(5000).copy()
movies_small = movies_small.reset_index(drop=True)

movies_small.head()

,movieId,title,genres,genres_clean
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,Adventure Animation Children Comedy Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy,Adventure Children Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance,Comedy Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,Comedy Drama Romance
4,5,Father of the Bride Part II (1995),Comedy,Comedy


In [5]:
# TF-IDF vectorization
tfidf = TfidfVectorizer(stop_words="english")

tfidf_matrix = tfidf.fit_transform(movies_small["genres_clean"].fillna(""))

In [6]:
# Cosine similarity
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [7]:
# Create index mapping: movie title -> row index
indices = pd.Series(
    movies_small.index,
    index=movies_small["title"]
).drop_duplicates()

In [8]:
def recommend_content_based(title, top_n=10):
    if title not in indices:
        return f"Movie '{title}' not found. Try exact title from movies.csv."

    idx = indices[title]

    sim_scores = list(enumerate(cosine_sim[idx]))

    sim_scores = sorted(
        sim_scores,
        key=lambda x: x[1],
        reverse=True
    )

    sim_scores = sim_scores[1:top_n + 1]

    movie_indices = [i[0] for i in sim_scores]

    result = movies_small.iloc[movie_indices][
        ["movieId", "title", "genres"]
    ].copy()

    result["similarity_score"] = [i[1] for i in sim_scores]

    return result

In [11]:
recommend_content_based("Toy Story (1995)", 10)

,movieId,title,genres,similarity_score
2203,2294,Antz (1998),Adventure|Animation|Children|Comedy|Fantasy,1.000000
3021,3114,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,1.000000
3653,3754,"Adventures of Rocky and Bullwinkle, The (2000)",Adventure|Animation|Children|Comedy|Fantasy,1.000000
3912,4016,"Emperor's New Groove, The (2000)",Adventure|Animation|Children|Comedy|Fantasy,1.000000
4780,4886,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy,1.000000
1944,2033,"Black Cauldron, The (1985)",Adventure|Animation|Children|Fantasy,0.967901
2026,2116,"Lord of the Rings, The (1978)",Adventure|Animation|Children|Fantasy,0.967901
3305,3400,We're Back! A Dinosaur's Story (1993),Adventure|Animation|Children|Fantasy,0.967901
4261,4366,Atlantis: The Lost Empire (2001),Adventure|Animation|Children|Fantasy,0.967901
4414,4519,"Land Before Time, The (1988)",Adventure|Animation|Children|Fantasy,0.967901
